# RetailPulse Phase 3.5 - Enterprise Forecasting Optimization

This notebook reuses the existing processed dataset and executes the upgraded enterprise forecasting engine. It compares the available model zoo, tunes leading candidates, generates business-facing multi-horizon forecasts, and captures the reasoning behind final model selection.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd().resolve().parent
sys.path.append(str(project_root / 'src'))

from forecasting_enhanced import run_enhanced_forecasting

result = run_enhanced_forecasting(
    input_path=project_root / 'data' / 'processed' / 'final_processed_dataset.csv',
    output_dir=project_root / 'processed',
    reports_dir=project_root / 'reports',
    figures_dir=project_root / 'reports' / 'figures',
    models_dir=project_root / 'models',
)

result['selected_series'][['SeriesKey', 'TotalSales', 'TotalRevenue', 'AvailableHistoryDays']].head()

## Model Comparison And Final Selection

Review the ranked leaderboard, the auto-selected winning model, and the rationale behind that choice. The enhanced pipeline keeps lower-coverage experiments visible in the leaderboard while restricting final auto-selection to models evaluated across the full selected-series portfolio.

In [ ]:
leaderboard = result['leaderboard'].copy()
best_summary = pd.DataFrame([
    {
        'best_model': result['best_model'],
        'best_configuration': result['best_configuration'],
        'selection_reason': result['best_reason'],
    }
])

display(best_summary)
display(leaderboard.head(15))

## Hyperparameter Tuning And Forecast Interpretation

The Phase 3.5 workflow applies recursive feature elimination plus grid or random search to the strongest tunable candidates. Use the tuning summary and feature-importance outputs below to understand why the winning model performs well.

In [ ]:
display(result['tuning_summary'].head(10))
display(result['feature_importance'].head(15))
display(result['permutation_importance'].head(15))
display(result['shap_summary'].head(15))

## Business Insights And Multi-Horizon Forecasts

The generated forecast tables support SKU, category, country, revenue, and demand planning across 30, 60, 90, 180, and 365-day horizons. The dashboard view below is useful for summarising forecast concentration and prioritisation.

In [ ]:
future_predictions = result['future_predictions'].copy()
dashboard = result['forecast_dashboard'].copy()

display(
    future_predictions[
        ['Date', 'ForecastLevel', 'Entity', 'HorizonDays', 'ForecastDemand', 'ForecastRevenue']
    ].head(20)
)
display(dashboard.head(20))

## Deep Learning Coverage

The enhanced engine includes TensorFlow/Keras hooks for LSTM, Bidirectional LSTM, GRU, and CNN-LSTM models with sequence windowing, normalization, EarlyStopping, ModelCheckpoint, and learning-rate scheduling. When TensorFlow is unavailable, the pipeline records that status and continues without failing.

In [ ]:
display(result['deep_learning_metrics'])
display(pd.DataFrame({'report': list(result['report_paths'].keys()), 'path': [str(path) for path in result['report_paths'].values()]}))